# DS23 · Session 46 · Multi-Agent Systems 3/3
## The Full Assembly and the Second Measurement

**BIU DS23 by Shlomit Levavi**

Everything we built across S42-S45 comes together here into one crew, built in LangGraph. Then the
honest measurement, **One Agent or a Crew**: a strong single agent versus a crew of three
(researcher, writer, judge), scored on quality, latency AND cost. The result is deliberately honest.

**The punchline stands:** *Autonomy is a budget, not a switch.* A crew must earn its ~3x cost.

**Zoom-safe:** every demo runs on a saved deterministic path (`USE_LIVE=False`).

### Setup: backend, pricing, the budget guard, and the latency model
Same spine as S42-S45, plus a small deterministic latency model so the measurement is reproducible offline.

In [1]:
import os, json, re, math
from typing import TypedDict, Annotated
import operator

SEED = 42
BACKEND = "groq"
MODEL   = "llama-3.3-70b-versatile"

MAX_ITERATIONS = 8
ROUND_CAP      = 1          # judge may send the report back to rewrite at most once (S45)
COST_CAP_USD   = 0.10
PRICE_IN  = 0.59 / 1_000_000
PRICE_OUT = 0.79 / 1_000_000

# deterministic latency model (offline), so the measurement is reproducible
MS_PER_CALL    = 800        # one API call
MS_PER_HANDOFF = 160        # one message passed between agents

USE_LIVE = False            # Zoom-safe: saved deterministic path. Flip to True after verifying the key.

def cost(cin, cout): return cin * PRICE_IN + cout * PRICE_OUT

print("guard:", MAX_ITERATIONS, "iters,", ROUND_CAP, "round cap,", COST_CAP_USD, "USD")

guard: 8 iters, 1 round cap, 0.1 USD


## Part A · The full assembly

### The canonical tools (from S42)
Defined here so the notebook is self-contained. Identical to the whole module.

In [2]:
def calculator(expression: str) -> str:
    """Evaluate an exact arithmetic expression."""
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return "error: unsupported characters"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"error: {e}"

_REG = {
    "gdpr breach":  ("GDPR Art.33: notify the authority within 72 hours of awareness.", [33]),
    "ai act risk":  ("AI Act Art.6: high-risk systems require conformity assessment.", [6]),
}
def search_regulation(query: str, regulation: str) -> str:
    """Retrieve a grounded passage from GDPR or the AI Act, with article ids."""
    for key, (text, ids) in _REG.items():
        if any(w in query.lower() for w in key.split()):
            return json.dumps({"passage": text, "article_ids": ids})
    return json.dumps({"passage": "no relevant passage", "article_ids": []})

TOOL_FNS = {"calculator": calculator, "search_regulation": search_regulation}
print("tools ready:", list(TOOL_FNS))

tools ready: ['calculator', 'search_regulation']


### The personas, as work contracts
Each role is a system prompt: a job description with one responsibility, its tools, and a constraint.
This is the S44 "persona as contract" idea, now the spec for the crew.

In [3]:
PERSONAS = {
    "researcher": {
        "role": "retrieve grounded facts from the regulation",
        "tools": ["search_regulation"],        # read-only: a safety boundary
        "constraint": "does not write the report; returns findings with article ids",
    },
    "writer": {
        "role": "draft the comparison report from the findings only",
        "tools": [],                            # no tools: cannot invent facts
        "constraint": "uses only the researcher's findings, never invents an article",
    },
    "judge": {
        "role": "score the draft against a pre-written rubric and decide",
        "tools": [],
        "constraint": "may send back to rewrite at most ROUND_CAP times",
    },
}
for name, p in PERSONAS.items():
    print(f"{name:<11} tools={p['tools']}  ::  {p['role']}")

researcher  tools=['search_regulation']  ::  retrieve grounded facts from the regulation
writer      tools=[]  ::  draft the comparison report from the findings only
judge       tools=[]  ::  score the draft against a pre-written rubric and decide


### Building the crew in LangGraph
The full graph: researcher to writer to judge, with a conditional edge. The judge either passes to
END or sends the draft back to the writer, at most `ROUND_CAP` times. Offline the node functions are
deterministic stubs; the graph structure is real LangGraph.

In [4]:
from langgraph.graph import StateGraph, END

class CrewState(TypedDict):
    task: str
    findings: str
    draft: str
    score: float
    rounds: int
    trace: Annotated[list, operator.add]

def researcher_node(s: CrewState):
    hit = json.loads(search_regulation("gdpr breach ai act risk", "both"))
    findings = "GDPR Art.33 (72h breach notice); AI Act Art.6 (high-risk conformity)."
    return {"findings": findings, "trace": [("researcher", "retrieved arts 33, 6")]}

def writer_node(s: CrewState):
    # a rewrite (round 2) is more grounded than the first draft
    grounded = s.get("rounds", 0) >= 1
    draft = ("Both regimes impose duties: GDPR Art.33 requires 72h breach notice; "
             "AI Act Art.6 requires conformity assessment for high-risk systems."
             + (" Every claim cites its article." if grounded else ""))
    return {"draft": draft, "trace": [("writer", "draft v%d" % (s.get("rounds",0)+1))]}

def judge_node(s: CrewState):
    grounded = "cites its article" in s["draft"]
    score = 0.90 if grounded else 0.74
    return {"score": score, "rounds": s.get("rounds", 0) + 1,
            "trace": [("judge", "scored %.2f" % score)]}

def judge_decision(s: CrewState):
    # pass if good enough, else rewrite, but never more than ROUND_CAP times
    if s["score"] >= 0.85 or s["rounds"] > ROUND_CAP:
        return "pass"
    return "rewrite"

g = StateGraph(CrewState)
g.add_node("researcher", researcher_node)
g.add_node("writer", writer_node)
g.add_node("judge", judge_node)
g.set_entry_point("researcher")
g.add_edge("researcher", "writer")
g.add_edge("writer", "judge")
g.add_conditional_edges("judge", judge_decision, {"pass": END, "writer": "writer", "rewrite": "writer"})
crew = g.compile()

result = crew.invoke({"task": "AI Act vs GDPR duties report", "rounds": 0, "trace": []})
print("protocol trace:")
for who, what in result["trace"]:
    print(f"  [{who:<11}] {what}")
print("final score:", result["score"])
assert result["score"] >= 0.85            # the crew reached the quality bar
assert result["rounds"] <= ROUND_CAP + 1  # the round cap held

protocol trace:
  [researcher ] retrieved arts 33, 6
  [writer     ] draft v1
  [judge      ] scored 0.74
  [writer     ] draft v2
  [judge      ] scored 0.90
final score: 0.9


### The guard node catches a staged budget overflow
In the graph, the budget guard is a node, not a side comment. We stage an overflow and watch it stop
the crew gracefully, the team-level version of MAX_ITERATIONS from S42.

In [5]:
def budget_guard(spent_usd, cap=COST_CAP_USD):
    """A guard node: stop the whole crew if cumulative cost exceeds the cap."""
    if spent_usd > cap:
        return {"stop": True, "answer": "Stopped: crew exceeded the budget cap."}
    return {"stop": False}

# normal run: under budget
print("under budget:", budget_guard(0.04))
# staged overflow: a long crew run blows the cap
staged = budget_guard(0.13)
print("over budget :", staged)
assert staged["stop"] is True
print("\nguard fired at the team level. 'Stopped' is a legitimate outcome, not a crash.")

under budget: {'stop': False}
over budget : {'stop': True, 'answer': 'Stopped: crew exceeded the budget cap.'}

guard fired at the team level. 'Stopped' is a legitimate outcome, not a crash.


## Part B · One Agent or a Crew

### The fair baseline and the metrics model
The baseline is a strong single agent with all tools and an excellent prompt, not a straw man.
We model cost (tokens) and latency (calls + handoffs) deterministically, and quality per rubric
criterion. Same honest-measurement treatment as The Loop Pays in S43.

In [6]:
def metrics(calls, handoffs, quality_by_criterion):
    cin  = sum(c for c, _ in calls)
    cout = sum(o for _, o in calls)
    return {
        "quality": sum(quality_by_criterion.values()) / len(quality_by_criterion),
        "by_criterion": quality_by_criterion,
        "n_calls": len(calls),
        "cost_per_report": cost(cin, cout),
        "latency_ms": len(calls) * MS_PER_CALL + handoffs * MS_PER_HANDOFF,
    }

# single agent: 3 calls, growing context, no handoffs. Strong, all tools, great prompt.
single = metrics(
    calls=[(420,110),(740,110),(1120,240)],
    handoffs=0,
    quality_by_criterion={"source_attribution":0.78, "structure":0.85, "completeness":0.83},
)
print("single agent: quality %.2f, %d calls, $%.5f/report, %dms"
      % (single["quality"], single["n_calls"], single["cost_per_report"], single["latency_ms"]))

single agent: quality 0.82, 3 calls, $0.00171/report, 2400ms


### The crew's metrics
The crew runs researcher (2 calls) to writer to judge, with one rewrite cycle. Each call re-sends
the persona and the handoff context, so the per-call tokens are larger. Three handoffs between roles.

In [7]:
crew_m = metrics(
    calls=[(600,60),(950,60),      # researcher x2
           (1260,240),             # writer
           (1420,120),             # judge
           (1500,220),             # writer rewrite
           (1650,100)],            # judge recheck
    handoffs=3,
    quality_by_criterion={"source_attribution":0.95, "structure":0.88, "completeness":0.87},
)
print("crew of 3:   quality %.2f, %d calls, $%.5f/report, %dms"
      % (crew_m["quality"], crew_m["n_calls"], crew_m["cost_per_report"], crew_m["latency_ms"]))

crew of 3:   quality 0.90, 6 calls, $0.00499/report, 5280ms


### The measurement: One Agent or a Crew
Three columns, per 1,000 reports. We assert on the SHAPE of the result, not the exact float:
the crew wins quality by a small margin and pays roughly 3x. Modeled, reproducible, honest.

In [8]:
def row(name, m):
    return (f"{name:<16}{m['quality']:>8.2f}{m['latency_ms']/single['latency_ms']:>9.1f}x"
            f"{'$'+format(m['cost_per_report']*1000,'.2f'):>13}")

print(f"{'approach':<16}{'quality':>8}{'latency':>10}{'cost/1k':>13}")
print(row("single agent", single))
print(row("crew of three", crew_m))

margin   = crew_m["quality"] - single["quality"]
cost_x   = crew_m["cost_per_report"] / single["cost_per_report"]
print(f"\nquality margin: +{margin:.2f}   cost multiple: {cost_x:.1f}x")

# Assert v3.1: shape, not exact floats
assert crew_m["quality"] > single["quality"]      # the crew wins quality
assert margin <= 0.12                               # but by a small margin
assert 2.5 <= cost_x <= 3.5                          # paying roughly 3x
print("\nHonest result: the crew wins quality by a small margin, and pays ~3x. Burden on the crew.")

approach         quality   latency      cost/1k
single agent        0.82      1.0x        $1.71
crew of three       0.90      2.2x        $4.99

quality margin: +0.08   cost multiple: 2.9x

Honest result: the crew wins quality by a small margin, and pays ~3x. Burden on the crew.


### Reading the result: where each one won
The average hides the story. We break quality down by rubric criterion: the crew earns its cost on
source attribution; on structure the single agent nearly matches it, at a third of the cost.

In [9]:
print(f"{'criterion':<20}{'single':>8}{'crew':>8}{'winner':>10}")
for crit in single["by_criterion"]:
    s = single["by_criterion"][crit]; c = crew_m["by_criterion"][crit]
    winner = "crew" if c - s > 0.05 else "tie"
    print(f"{crit:<20}{s:>8.2f}{c:>8.2f}{winner:>10}")

print("\n- crew wins big on source_attribution (specialization + judge caught missing citations)")
print("- near-tie on structure: the single agent matched it at ~1/3 the cost")
# the crew's edge is concentrated, not uniform
assert crew_m["by_criterion"]["source_attribution"] - single["by_criterion"]["source_attribution"] > 0.1
assert abs(crew_m["by_criterion"]["structure"] - single["by_criterion"]["structure"]) <= 0.05
print("\nUse a crew where its edge is concentrated and the task needs it, not by default.")

criterion             single    crew    winner
source_attribution      0.78    0.95      crew
structure               0.85    0.88       tie
completeness            0.83    0.87       tie

- crew wins big on source_attribution (specialization + judge caught missing citations)
- near-tie on structure: the single agent matched it at ~1/3 the cost

Use a crew where its edge is concentrated and the task needs it, not by default.


## Transfer to your data (home)
Step 4 of the assignment: add the budget guard to your agent and begin the trajectory evaluation,
10 tasks you define including 2 the agent fails, with a written analysis of each failure.

In [10]:
# TODO (home, Step 4): lock a budget guard (MAX_ITERATIONS + COST_CAP) into your agent,
# then start the trajectory eval: 10 tasks, including 2 the agent fails.
# For each failure, write: which iteration, which tool was called, and why it was not enough.
print("Your agent, measured. See the assignment brief for Step 4.")

Your agent, measured. See the assignment brief for Step 4.
